# Chapitre 6 — Structuration et transformation

**Durée estimée : 8-10 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Restructurer** des DataFrames avec pivot et melt selon les besoins d'analyse
2. **Combiner** des sources de données avec merge et concat en choisissant la bonne méthode
3. **Agréger** des données avec groupby et créer des statistiques par groupe
4. **Créer** de nouvelles variables pertinentes (feature engineering) pour l'analyse et le ML

---

## 🎯 Le Hook : Comment Spotify crée 10 000 features pour vous connaître

Spotify ne stocke pas simplement "vous avez écouté cette chanson". À partir de vos données d'écoute brutes, ils créent plus de **10 000 features** par utilisateur :

- Nombre d'écoutes par genre, par heure, par jour
- Tempo moyen préféré
- Diversité musicale (ratio nouveaux artistes / artistes habituels)
- Patterns saisonniers (musique de Noël en décembre ?)

Ces features transformées alimentent ensuite l'algorithme de recommandation.

La donnée brute ("user_123 a écouté track_456 à 14h32") devient une **représentation riche** de vos goûts.

C'est le pouvoir de la **transformation des données**.

> 💭 **Question Socratique #1** : Si Spotify stocke 10 000 features par utilisateur pour 500 millions d'utilisateurs, cela représente des milliards de données. Est-ce vraiment nécessaire de tout stocker, ou pourrait-on recalculer à la demande ?

> 💭 **Réponse** 

**Non, il serait techniquement et économiquement impossible de tout recalculer à la demande.**

Voici pourquoi Spotify (et les géants de la tech) choisissent de transformer et stocker ces 10 000 features plutôt que de repartir de zéro à chaque fois :

---
1. Le problème de la latence (L'expérience utilisateur)

Si vous ouvrez votre application et que l'algorithme doit scanner l'intégralité de votre historique d'écoute (des années de données brutes) pour calculer votre "tempérament musical du moment" avant de proposer une playlist, vous attendriez plusieurs minutes.

* **Stockage = Rapidité :** En stockant des features pré-calculées, Spotify peut servir des recommandations en quelques millisecondes.

2. Le coût du calcul (Compute Cost)

Recalculer 10 000 variables pour 500 millions d'utilisateurs à chaque interaction coûterait une fortune en serveurs.

* **L'approche "Batch" :** Il est beaucoup plus efficace de faire tourner un immense calcul une fois par jour (ou par heure) qui met à jour votre profil, puis de stocker ce résultat "statique". Le stockage coûte aujourd'hui bien moins cher que le processeur (CPU/GPU).

3. La hiérarchie de la donnée

Dans la pratique, Spotify utilise une approche hybride que l'on appelle souvent la **Lambda Architecture** :

| Type de donnée | Méthode | Utilité |
| --- | --- | --- |
| **Historique long terme** | Stocké (Pre-computed) | Connaître vos goûts profonds (Jazz, Rock, années 80). |
| **Donnée temps réel** | Calculé à la volée | S'adapter à ce que vous écoutez *maintenant* (ex: vous êtes à la salle de sport). |

---

4. L'entraînement des modèles de Machine Learning

Pour que l'algorithme apprenne, il a besoin de voir des tendances. On ne peut pas entraîner une IA sur des "clics" isolés ; elle a besoin de ces **10 000 features** (votre ADN musical) pour comparer votre profil à celui de millions d'autres utilisateurs similaires ("Collaborative Filtering").

Le saviez-vous ?

Spotify utilise un outil interne appelé **Feast** (Feature Store). C'est comme une immense bibliothèque où les data scientists rangent ces 10 000 features bien étiquetées pour qu'elles soient réutilisables par n'importe quel algorithme de la plateforme sans avoir à tout réinventer.

---

# 📖 PARTIE THÉORIQUE

---

## 6.1 Restructuration des DataFrames

### Wide vs Long : deux visions des mêmes données

```
FORMAT WIDE (large)                    FORMAT LONG (long)
┌────────┬─────┬─────┬─────┐          ┌────────┬───────┬────────┐
│ Client │ Jan │ Fev │ Mar │          │ Client │ Mois  │ Ventes │
├────────┼─────┼─────┼─────┤          ├────────┼───────┼────────┤
│ Alice  │ 100 │ 150 │ 120 │    ↔     │ Alice  │ Jan   │ 100    │
│ Bob    │ 200 │ 180 │ 220 │          │ Alice  │ Fev   │ 150    │
└────────┴─────┴─────┴─────┘          │ Alice  │ Mar   │ 120    │
                                      │ Bob    │ Jan   │ 200    │
1 ligne par client                    │ Bob    │ Fev   │ 180    │
Colonnes = périodes                   │ Bob    │ Mar   │ 220    │
                                      └────────┴───────┴────────┘
                                      1 ligne par observation
```

### Quand utiliser quel format ?

| Format | Avantages | Cas d'usage |
|--------|-----------|-------------|
| **Wide** | Lisible, compact | Tableaux de reporting, Excel |
| **Long** | Flexible, analyses faciles | Visualisation, ML, base de données |

### Types de jointures (merge)

```
       INNER                LEFT                RIGHT               OUTER
    ┌─────────┐          ┌─────────┐          ┌─────────┐          ┌─────────┐
    │    A    │          │    A    │          │         │          │    A    │
    │  ┌───┐  │          │  ┌───┐  │          │  ┌───┐  │          │  ┌───┐  │
    │  │ X │  │          │  │ X │  │          │  │ X │  │          │  │ X │  │
    │  └───┘  │          │  └───┘  │          │  └───┘  │          │  └───┘  │
    │    B    │          │         │          │    B    │          │    B    │
    └─────────┘          └─────────┘          └─────────┘          └─────────┘

   Seulement ce        Tout A +            Tout B +             Tout A + Tout B
   qui matche          matchs de B         matchs de A
```

### Merge vs Concat

| Critère | Merge | Concat |
|---------|-------|--------|
| **Utilisation** | Joindre sur une clé | Empiler sans condition |
| **Colonnes** | Peuvent être différentes | Doivent correspondre (axis=0) |
| **Analogie SQL** | JOIN | UNION |
| **Cas typique** | Clients + Commandes | Janvier + Février + Mars |

### Le paradigme Split-Apply-Combine (GroupBy)

```
        DONNÉES ORIGINALES
        ┌──────────────────┐
        │ Region  Ventes   │
        │ Nord    100      │
        │ Sud     150      │
        │ Nord    120      │
        │ Sud     180      │
        │ Nord    90       │
        └──────────────────┘
              │ SPLIT
              ▼
    ┌─────────────┬─────────────┐
    │   Nord      │    Sud      │
    │   100       │    150      │
    │   120       │    180      │
    │   90        │             │
    └─────────────┴─────────────┘
              │ APPLY (sum)
              ▼
    ┌─────────────┬─────────────┐
    │   310       │    330      │
    └─────────────┴─────────────┘
              │ COMBINE
              ▼
        ┌──────────────────┐
        │ Region  Total    │
        │ Nord    310      │
        │ Sud     330      │
        └──────────────────┘
```